# 2 — Extractor TensorRT export

**Prerequisites:**
- `uv sync --group trt` (or `--all-groups`) for `polygraphy`, `tensorrt`
- `pip install onnxsim` (or add `onnxsim` to the `trt` dependency group in `pyproject.toml`)
- A CUDA-capable GPU with TensorRT installed

**Prerequisite:** Run [1-export_extractors.ipynb](1-export_extractors.ipynb) first to generate the `.onnx` files in `weights/euroc/`.

In [ ]:
from pathlib import Path

CWD = Path.cwd().resolve()
if CWD.name == "notebooks":
    LG = CWD.parent
elif CWD.name == "LightGlue-ONNX-Jetson":
    LG = CWD
else:
    raise SystemExit("Run Jupyter with cwd = LightGlue-ONNX-Jetson or LightGlue-ONNX-Jetson/notebooks")

OUT_DIR = LG / "weights" / "euroc"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("LG     ", LG)
print("OUT_DIR", OUT_DIR, OUT_DIR.exists())

### ONNX models to process

In [ ]:
# List only the original (non-simplified) ONNX models
onnx_models = sorted(p for p in OUT_DIR.glob("*.onnx") if "_sim" not in p.stem)
for p in onnx_models:
    print(p.name, f"  {p.stat().st_size / 1e6:.1f} MB")

if not onnx_models:
    raise SystemExit("No ONNX models found in OUT_DIR. Run export_extractors.ipynb first.")

### ONNX simplification (onnxsim)

Runs `onnxsim` on each model. Constant folding and dead-node elimination reduce graph size
and improve TensorRT compilation speed. Output: `*_sim.onnx` alongside the originals.

In [ ]:
import sys

if str(LG) not in sys.path:
    sys.path.insert(0, str(LG))

from lightglue_dynamo.extractor_export import simplify_onnx

sim_paths: dict[str, Path] = {}
for onnx_path in onnx_models:
    sim_path = onnx_path.with_name(onnx_path.stem + "_sim.onnx")
    try:
        simplify_onnx(onnx_path, sim_path)
        ratio = sim_path.stat().st_size / onnx_path.stat().st_size
        # Growth >10% is typically DeformConv Constant materialisation — expected and correct.
        flag = " [grew — Constant→initializer materialisation, check=True]" if ratio > 1.1 else ""
        print(f"[OK]   {onnx_path.name}")
        print(f"       -> {sim_path.name}  ({ratio:.1%} of original){flag}")
        sim_paths[onnx_path.stem] = sim_path
    except Exception as exc:
        print(f"[ERR]  {onnx_path.name}: {exc} — using original")
        sim_paths[onnx_path.stem] = onnx_path

print(f"\nSimplified {sum(1 for p in sim_paths.values() if '_sim' in p.stem)}/{len(onnx_models)} models.")

### TensorRT engine build (FP16)

Builds one FP16 engine per simplified ONNX. Engines are saved as `*_sim.fp16.engine`.

> **ALIKED note:** ALIKED uses `DeformConv` (ONNX opset 19).

In [ ]:
from lightglue_dynamo.extractor_export import build_extractor_trt_engine

engine_paths: dict[str, Path] = {}
for stem, sim_path in sim_paths.items():
    engine_path = sim_path.with_suffix(".fp16.engine")
    try:
        build_extractor_trt_engine(sim_path, engine_path, fp16=True)
        print(f"[OK]   {engine_path.name}  ({engine_path.stat().st_size / 1e6:.1f} MB)")
        engine_paths[stem] = engine_path
    except Exception as exc:
        print(f"[FAIL] {stem}: {exc}")

print(f"\nBuilt {len(engine_paths)}/{len(sim_paths)} TRT engines.")

### Test — ORT CUDA vs TRT

Runs both ORT CUDA (on the simplified ONNX) and TRT (on the engine) with the same
test images. Reports output shapes and max absolute difference for float tensors.

In [ ]:
from lightglue_dynamo.extractor_export import EXTRACTOR_REGISTRY


def stem_to_extractor_id(stem: str) -> str | None:
    """Infer extractor id from a filename stem, e.g. 'superpoint_open_b2_...' -> 'superpoint_open'."""
    for eid in sorted(EXTRACTOR_REGISTRY, key=len, reverse=True):
        if stem.startswith(eid):
            return eid
    return None

In [ ]:
import re
import time

import cv2
import numpy as np
import onnxruntime as ort
from polygraphy.backend.common import BytesFromPath
from polygraphy.backend.trt import EngineFromBytes, TrtRunner

PROVIDERS_ORT = ["CUDAExecutionProvider", "CPUExecutionProvider"]
N_WARMUP = 5
N_TIMED  = 20
asset_dir = LG / "assets"
img_files = ["sacre_coeur1.jpg", "sacre_coeur2.jpg"]

def parse_hw_from_stem(stem: str) -> tuple[int, int]:
    m = re.search(r"_h(\d+)_w(\d+)", stem)
    if m is None:
        raise ValueError(f"Cannot parse H/W from model stem: {stem}")
    return int(m.group(1)), int(m.group(2))


def load_imgs_bgr(h: int, w: int) -> list[np.ndarray]:
    imgs = []
    for f in img_files:
        p = asset_dir / f
        if p.is_file():
            imgs.append(cv2.resize(cv2.imread(str(p)), (w, h)))
        else:
            print(f"[WARN] {f} not found — using random noise image")
            imgs.append(np.random.randint(0, 255, (h, w, 3), dtype=np.uint8))
    return imgs


def make_batch(spec, imgs):
    """Preprocess a list of images into (B, C, H, W), handling preprocessors
    that return (C,H,W) or already-batched (1,C,H,W)."""
    frames = []
    for img in imgs:
        p = spec.preprocess(img)
        while p.ndim > 3:
            p = p[0]
        frames.append(p)
    return np.stack(frames, axis=0)


for stem, engine_path in engine_paths.items():
    sim_path = sim_paths[stem]
    eid = stem_to_extractor_id(stem)
    if eid is None:
        print(f"[SKIP] {stem}: cannot resolve extractor id")
        continue

    h, w = parse_hw_from_stem(stem)
    imgs_bgr = load_imgs_bgr(h, w)

    spec = EXTRACTOR_REGISTRY[eid]
    x = make_batch(spec, imgs_bgr)

    # ORT reference — try CUDA EP first, fall back to CPU; skip if ORT lacks the op (e.g. DeformConv)
    ort_outs = None
    ort_ep = None
    last_ort_exc = None
    for providers in [PROVIDERS_ORT, ["CPUExecutionProvider"]]:
        try:
            sess = ort.InferenceSession(str(sim_path), providers=providers)
            ort_outs = sess.run(None, {spec.input_name: x})
            ort_ep = providers[0].replace("ExecutionProvider", "")
            break
        except Exception as exc:
            last_ort_exc = exc

    # TRT — warmup then timed runs
    with TrtRunner(EngineFromBytes(BytesFromPath(str(engine_path)))) as runner:
        for _ in range(N_WARMUP):
            runner.infer(feed_dict={spec.input_name: x})
        t0 = time.perf_counter()
        for _ in range(N_TIMED):
            trt_outs = runner.infer(feed_dict={spec.input_name: x})
        elapsed_ms = (time.perf_counter() - t0) / N_TIMED * 1000

    label = f"ORT ref: SKIP" if ort_outs is None else f"ORT ref: {ort_ep}"
    print(f"\n=== {eid} (H={h}, W={w})  ({label})  TRT {elapsed_ms:.2f} ms/batch  ({elapsed_ms/len(imgs_bgr):.2f} ms/img, B={len(imgs_bgr)}) ===")

    if ort_outs is None:
        print(f"  skip reason: {last_ort_exc}")
        print(f"  TRT outputs (shapes only):")
        for name, arr in trt_outs.items():
            print(f"  {name}: shape={arr.shape}  dtype={arr.dtype}")
    else:
        for i, name in enumerate(spec.output_names):
            ort_t = ort_outs[i]
            trt_t = trt_outs.get(name)
            if trt_t is None:
                print(f"  {name}: TRT output missing")
                continue
            if np.issubdtype(ort_t.dtype, np.floating):
                max_diff = float(np.abs(ort_t.astype(np.float32) - trt_t.astype(np.float32)).max())
                print(f"  {name}: shape={ort_t.shape}  dtype={ort_t.dtype}  max_diff={max_diff:.5f}")
            else:
                match = np.array_equal(ort_t, trt_t)
                print(f"  {name}: shape={ort_t.shape}  dtype={ort_t.dtype}  exact_match={match}")

### Keypoint visualisation (TRT outputs)

In [ ]:
import matplotlib.pyplot as plt

n_rows = len(engine_paths)
n_cols = len(img_files)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 4 * n_rows), dpi=96)
if n_rows == 1:
    axes = np.expand_dims(axes, 0)
fig.suptitle("TRT Extractor Smoke Test  (per-model H/W from engine name, FP16)", fontsize=13)


def to_pixel_coords(kpts: np.ndarray, h: int, w: int) -> np.ndarray:
    """Denormalize keypoints from [-1, 1] to pixel space if needed."""
    if np.all((kpts >= -1.1) & (kpts <= 1.1)):
        wh = np.array([w - 1, h - 1], dtype=np.float32)
        return (kpts + 1) / 2 * wh
    return kpts


for row_i, (stem, engine_path) in enumerate(engine_paths.items()):
    eid = stem_to_extractor_id(stem)
    spec = EXTRACTOR_REGISTRY[eid]

    h, w = parse_hw_from_stem(stem)
    imgs_bgr = load_imgs_bgr(h, w)
    x = make_batch(spec, imgs_bgr)  # (B, C, H, W) — handles grayscale and RGB

    with TrtRunner(EngineFromBytes(BytesFromPath(str(engine_path)))) as runner:
        trt_outs = runner.infer(feed_dict={spec.input_name: x})

    kpts_b = trt_outs["keypoints"]          # (B, K, 2)
    scores_b = trt_outs["keypoint_scores"]  # (B, K)
    num_kpts = trt_outs.get("num_keypoints")
    num_valid = [int(num_kpts[i]) if num_kpts is not None else int((scores_b[i] > 0).sum())
                 for i in range(len(imgs_bgr))]

    for col_i, img_bgr in enumerate(imgs_bgr):
        ax = axes[row_i, col_i]
        n = num_valid[col_i]
        h_img, w_img = img_bgr.shape[:2]
        ax.imshow(img_bgr[..., ::-1])
        if n > 0:
            kpts = to_pixel_coords(kpts_b[col_i, :n], h_img, w_img)
            scores = scores_b[col_i, :n]
            sc = ax.scatter(kpts[:, 0], kpts[:, 1], c=scores,
                            cmap="plasma", s=4, linewidths=2,
                            vmin=scores.min(), vmax=scores.max())
            plt.colorbar(sc, ax=ax, fraction=0.03, pad=0.02)
        ax.set_title(f"{eid} (H={h},W={w})  |  {img_files[col_i]}  ({n} kpts)", fontsize=8)
        ax.axis("off")

plt.tight_layout()
plt.show()